In [ ]:
import numpy as np

# 1. Convolution Layer
class Conv3x3:
    # A layer using 8 filters of size 3x3
    def __init__(self, num_filters):
        self.num_filters = num_filters
        # Filters are initialized with random values
        self.filters = np.random.randn(num_filters, 3, 3) / 9

    def iterate_regions(self, image):
        # Generates all possible 3x3 image regions
        h, w = image.shape
        for i in range(h - 2):
            for j in range(w - 2):
                im_region = image[i:(i + 3), j:(j + 3)]
                yield im_region, i, j

    def forward(self, input):
        self.last_input = input
        h, w = input.shape
        output = np.zeros((h - 2, w - 2, self.num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.sum(im_region * self.filters, axis=(1, 2))
        return output

    def backprop(self, d_L_d_out, learn_rate):
        # d_L_d_out is the loss gradient for this layer's outputs
        d_L_d_filters = np.zeros(self.filters.shape)

        for im_region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.num_filters):
                d_L_d_filters[f] += d_L_d_out[i, j, f] * im_region

        # Update filters
        self.filters -= learn_rate * d_L_d_filters
        return None

# 2. Max Pooling Layer (2x2)
class MaxPool2:
    def iterate_regions(self, image):
        h, w, _ = image.shape
        new_h = h // 2
        new_w = w // 2
        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i * 2):(i * 2 + 2), (j * 2):(j * 2 + 2)]
                yield im_region, i, j

    def forward(self, input):
        self.last_input = input
        h, w, num_filters = input.shape
        output = np.zeros((h // 2, w // 2, num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.amax(im_region, axis=(0, 1))
        return output

    def backprop(self, d_L_d_out):
        d_L_d_input = np.zeros(self.last_input.shape)

        for im_region, i, j in self.iterate_regions(self.last_input):
            h, w, f = im_region.shape
            amax = np.amax(im_region, axis=(0, 1))

            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        # If this pixel was the max, copy the gradient to it
                        if im_region[i2, j2, f2] == amax[f2]:
                            d_L_d_input[i * 2 + i2, j * 2 + j2, f2] = d_L_d_out[i, j, f2]
        return d_L_d_input

# 3. Softmax / Fully Connected Layer
class Softmax:
    def __init__(self, input_len, nodes):
        # input_len: total pixels after flattening (13 * 13 * 8)
        # nodes: number of classes (10 for digits 0-9)
        self.weights = np.random.randn(input_len, nodes) / input_len
        self.biases = np.zeros(nodes)

    def forward(self, input):
        self.last_input_shape = input.shape
        input = input.flatten()
        self.last_input = input

        totals = np.dot(input, self.weights) + self.biases
        self.last_totals = totals

        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)

    def backprop(self, d_L_d_out, learn_rate):
        for i, gradient in enumerate(d_L_d_out):
            if gradient == 0: continue

            t_exp = np.exp(self.last_totals)
            S = np.sum(t_exp)

            # Gradients of out[i] wrt totals
            d_out_d_t = -t_exp[i] * t_exp / (S**2)
            d_out_d_t[i] = t_exp[i] * (S - t_exp[i]) / (S**2)

            # Gradients of loss wrt totals
            d_L_d_t = gradient * d_out_d_t

            # Gradients of loss wrt weights/biases/input
            d_L_d_w = self.last_input[np.newaxis].T @ d_L_d_t[np.newaxis]
            d_L_d_b = d_L_d_t
            d_L_d_inputs = self.weights @ d_L_d_t

            # Update weights and biases
            self.weights -= learn_rate * d_L_d_w
            self.biases -= learn_rate * d_L_d_b
            return d_L_d_inputs.reshape(self.last_input_shape)

# --- TRAINING LOOP ---
# (Assuming you have loaded MNIST data into train_images and train_labels)
# For simplicity, we use 1000 images for a quick demo
from keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

conv = Conv3x3(8)                  # 28x28 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13 * 13 * 8, 10) # 13x13x8 -> 10

def forward(image, label):
    # Normalize image from [0, 255] to [-0.5, 0.5]
    out = conv.forward((image / 255) - 0.5)
    out = pool.forward(out)
    out = softmax.forward(out)

    # Calculate Cross-Entropy Loss and Accuracy
    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0
    return out, loss, acc

def train(im, label, lr=0.005):
    # Forward Pass
    out, loss, acc = forward(im, label)
    # Initial Gradient
    gradient = np.zeros(10)
    gradient[label] = -1 / out[label]
    # Backward Pass
    gradient = softmax.backprop(gradient, lr)
    gradient = pool.backprop(gradient)
    conv.backprop(gradient, lr)
    return loss, acc

print('Starting training...')
for epoch in range(3):
    print(f'--- Epoch {epoch + 1} ---')
    loss = 0
    num_correct = 0
    for i, (im, label) in enumerate(zip(train_images[:1000], train_labels[:1000])):
        l, acc = train(im, label)
        loss += l
        num_correct += acc
    print(f'Loss: {loss/1000:.3f} | Accuracy: {num_correct/1000:.3f}')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Starting training...
--- Epoch 1 ---
Loss: 1.183 | Accuracy: 0.633
--- Epoch 2 ---
Loss: 0.517 | Accuracy: 0.845
--- Epoch 3 ---
Loss: 0.409 | Accuracy: 0.880


In [ ]:
def print_model_summary(input_shape=(28, 28)):
    # Initialize dummy data
    dummy_input = np.zeros(input_shape)

    print(f"{'Layer':<20} | {'Output Shape':<15} | {'Parameters':<10}")
    print("-" * 50)

    # Conv Layer
    out_conv = conv.forward((dummy_input / 255) - 0.5)
    print(f"{'Conv3x3':<20} | {str(out_conv.shape):<15} | {conv.filters.size:<10}")

    # Pool Layer
    out_pool = pool.forward(out_conv)
    print(f"{'MaxPool2':<20} | {str(out_pool.shape):<15} | {'0':<10}")

    # Softmax Layer
    out_soft = softmax.forward(out_pool)
    params = softmax.weights.size + softmax.biases.size
    print(f"{'Softmax':<20} | {str(out_soft.shape):<15} | {params:<10}")

# Call it after initializing your layers
print_model_summary()

Layer                | Output Shape    | Parameters
--------------------------------------------------
Conv3x3              | (26, 26, 8)     | 72        
MaxPool2             | (13, 13, 8)     | 0         
Softmax              | (10,)           | 13530     
